In [0]:
from pyspark.sql.functions import current_timestamp

In [0]:
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.dropdown("load_type", "FULL", ["FULL", "APPEND", "MERGE"], "Load Type")
dbutils.widgets.text("merge_keys", "", "Merge Keys (comma-separated)")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type").strip().upper()
merge_keys = [k.strip() for k in dbutils.widgets.get("merge_keys").split(",") if k.strip()]

if load_type not in ("FULL", "APPEND", "MERGE"):
    raise ValueError(f"Invalid load_type '{load_type}'. Expected FULL, APPEND or MERGE.")
if load_type == "MERGE" and not merge_keys:
    raise ValueError("merge_keys must be provided when load_type = 'MERGE'.")

In [0]:
bronze_df = spark.read.table(f"adb_cyrnes.bronze.{table_name}").filter(f"landing_timestamp = '{landing_timestamp}'")

In [0]:
new_df = bronze_df.withColumn("insert_timestamp",current_timestamp()).withColumn("update_timestamp",current_timestamp())

In [0]:
from delta.tables import DeltaTable

target_table = f"adb_cyrnes.silver.{table_name}"

if load_type == "FULL":
    # Full load: replace the target table contents
    new_df.write.mode("overwrite").saveAsTable(target_table)

elif load_type == "APPEND":
    # Append: add incoming records to the target table
    new_df.write.mode("append").saveAsTable(target_table)

elif load_type == "MERGE":
    missing = [k for k in merge_keys if k not in new_df.columns]
    if missing:
        raise ValueError(f"Merge keys {missing} not found in source columns: {new_df.columns}")

    if not spark.catalog.tableExists(target_table):
        # First run: target does not exist yet, so create it from the source
        new_df.write.mode("overwrite").saveAsTable(target_table)
    else:
        # A MERGE fails if several source rows match one target row, so keep one row per key
        source_df = new_df.dropDuplicates(merge_keys)

        merge_condition = " AND ".join([f"t.`{k}` = s.`{k}`" for k in merge_keys])
        # On update: keep the original insert_timestamp, refresh all data columns,
        # and stamp update_timestamp with the time of this merge
        update_set = {
            f"`{c}`": f"s.`{c}`"
            for c in source_df.columns
            if c not in ("insert_timestamp", "update_timestamp")
        }
        update_set["update_timestamp"] = "current_timestamp()"

        (
            DeltaTable.forName(spark, target_table).alias("t")
            .merge(source_df.alias("s"), merge_condition)
            .whenMatchedUpdate(set=update_set)
            .whenNotMatchedInsertAll()
            .execute()
        )

print(f"{load_type} load completed for {target_table}")

In [0]:
record_count = new_df.count()
dbutils.notebook.exit(str(record_count))